In [ ]:
import cv2
import mediapipe as mp
import serial
import time

# Configura el puerto serial (cambia 'COM4' por el puerto correcto)
arduino = serial.Serial('COM46', 9600)
time.sleep(2)  # Espera a que Arduino se inicialice

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

with mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Dibuja las landmarks
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # Palma - landmark 0
                palm = hand_landmarks.landmark[0]
                palm_x = palm.x
                palm_y = palm.y
                palm_z = palm.z

                # Dedo índice - punta (landmark 8)
                index_tip = hand_landmarks.landmark[8]
                index_x, index_y, index_z = index_tip.x, index_tip.y, index_tip.z

                # Pulgar - punta (landmark 4)
                thumb_tip = hand_landmarks.landmark[4]
                thumb_x, thumb_y, thumb_z = thumb_tip.x, thumb_tip.y, thumb_tip.z

                # Mostrar coordenadas en pantalla
                cv2.putText(image, f"Palma X: {palm_x:.3f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                cv2.putText(image, f"Palma Y: {palm_y:.3f}", (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                cv2.putText(image, f"Palma Z: {palm_z:.3f}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

                cv2.putText(image, f"Indice X: {index_x:.3f}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(image, f"Indice Y: {index_y:.3f}", (10, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(image, f"Indice Z: {index_z:.3f}", (10, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                cv2.putText(image, f"Pulgar X: {thumb_x:.3f}", (10, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
                cv2.putText(image, f"Pulgar Y: {thumb_y:.3f}", (10, 215), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
                cv2.putText(image, f"Pulgar Z: {thumb_z:.3f}", (10, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

                # Convertir palma X (0-1) a ángulo base servo (0-180)
                base_angle = int(palm_x * 180)
                base_angle = max(0, min(180, base_angle))
                arduino.write(f"1 {base_angle}\n".encode())

                # Convertir palma Z a ángulo servo extensión (90-180)
                z_normalized = min(max(palm_z, -0.2), 0.1)
                extent_angle = int(180 - ((z_normalized + 0.2) * (90 / 0.3)))
                extent_angle = max(90, min(180, extent_angle))
                arduino.write(f"3 {extent_angle}\n".encode())

                # Convertir distancia entre pulgar e índice para abrir/cerrar pinzas (servo 2)
                # Calculamos distancia euclidiana en 3D
                dx = thumb_x - index_x
                dy = thumb_y - index_y
                dz = thumb_z - index_z
                distance = (dx**2 + dy**2 + dz**2)**0.5

                # Mapear distancia a ángulo 50 (cerrado) - 110 (abierto)
                # Ajusta min_dist y max_dist según pruebas
                min_dist = 0.02
                max_dist = 0.15
                dist_clamped = max(min_dist, min(max_dist, distance))
                tweezers_angle = int(110 - (dist_clamped - min_dist) / (max_dist - min_dist) * 60)
                tweezers_angle = max(50, min(110, tweezers_angle))
                arduino.write(f"2 {tweezers_angle}\n".encode())

                # Mostrar ángulos en pantalla
                cv2.putText(image, f"Base Angle: {base_angle}", (400, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                cv2.putText(image, f"Extent Angle: {extent_angle}", (400, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                cv2.putText(image, f"Tweezers Angle: {tweezers_angle}", (400, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        cv2.imshow("Control Brazo Robot", image)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()
arduino.close()


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
pip install mediapipe opencv-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.12.0 requires protobuf<6.0.0,>=5.29.1, but you have protobuf 4.25.8 which is inco

In [ ]:
pip install pyserial


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.2 MB/s eta 0:00:00
